# GOTOken oracle

Reference values from the HuggingFace `SmolLM2-135M` model for each step's checkpoint,
and comparisons against the BASIC engine. All logic lives in `oracle.py`; this notebook
is the interactive front-end. Run `./build.sh` in the repo root first if you want the
BASIC comparisons.

In [1]:
from oracle import *
tok = load_tokenizer()
model = load_model()
model.config

/Users/bmuskalla/git/GOTOken/export/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/272 [00:00<00:40,  6.74it/s]

Loading weights:  71%|███████   | 193/272 [00:00<00:00, 928.00it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 805.56it/s]

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "dtype": "float32",
  "eos_token_id": 0,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 576,
  "initializer_range": 0.041666666666666664,
  "intermediate_size": 1536,
  "is_llama_config": true,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 9,
  "num_hidden_layers": 30,
  "num_key_value_heads": 3,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_interleaved": false,
  "rope_parameters": {
    "rope_theta": 100000,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.17.0",
  "use_cache": true,
  "vocab_size": 49152
}

## Step 2: embedding row -> tied output head

No norm, no attention, no FFN. With tied weights every logit is the dot product of the
input token's embedding with one vocab row, so the "prediction" is simply the nearest
embeddings. Note what the top-5 for ` cat` looks like.

In [2]:
for i in tok.encode("The cat sat", add_special_tokens=False):
    print(i, repr(tok.convert_ids_to_tokens(i)))

504 'The'
2644 'Ġcat'
2643 'Ġsat'


In [3]:
_ = print_step2(model, tok, 2644)

token 2644 = 'Ġcat'
logit 0 0.133899
logit 1 1.771975
logit 2 1.769763
logit 3 1.899459
logit 4 1.86695
top 1 id 2644 logit 3.985926   'Ġcat'
top 2 id 9786 logit 3.531259   'cat'
top 3 id 40578 logit 3.432986   'cats'
top 4 id 27772 logit 3.333802   'Cat'
top 5 id 6 logit 3.297556   '<filename>'


In [4]:
compare_step2(model, tok, 2644)


compare BASIC vs oracle
  logit 0: basic 0.133899  oracle 0.133899  diff 6.12e-09  ok
  logit 1: basic 1.771975  oracle 1.771975  diff 4.50e-07  ok
  logit 2: basic 1.769764  oracle 1.769763  diff 7.33e-07  ok
  logit 3: basic 1.899459  oracle 1.899459  diff 3.95e-07  ok
  logit 4: basic 1.86695  oracle 1.86695  diff 1.35e-07  ok
  top 1: basic id 2644 3.985922  oracle id 2644 3.985926  diff 3.90e-06  ok
  top 2: basic id 9786 3.531259  oracle id 9786 3.531259  diff 1.81e-07  ok
  top 3: basic id 40578 3.432987  oracle id 40578 3.432986  diff 8.10e-07  ok
  top 4: basic id 27772 3.333801  oracle id 27772 3.333802  diff 8.67e-07  ok
  top 5: basic id 6 3.297555  oracle id 6 3.297556  diff 7.97e-07  ok
PASS


True

## Step 3: the kernels in isolation

RMSNorm and MatMul on real layer-0 weights, fed the embedding row of ` cat`. The float64
result is the reference; the pass criterion is `|basic - ref| <= 1e-5 + 1e-5 * |ref|`.

The second table is the lesson: the same fp32 matmul computed four ways agrees with
BASIC bit for bit only when the accumulation order is replayed exactly. Nothing else
does, not even torch, and none of them is "wrong".

In [5]:
compare_step3(model, tok, 2644)


compare BASIC vs float64 oracle, token 2644 = 'Ġcat'
  criterion: |basic - ref| <= 1e-05 + 1e-05 * |ref|
  x        n= 576  max abs err 0.00e+00  max rel err 0.00e+00  bit-exact vs float64 576/576  ok
  rmsnorm  n= 576  max abs err 4.84e-07  max rel err 5.38e-07  bit-exact vs float64 0/576  ok
  wq       n= 576  max abs err 3.90e-06  max rel err 8.49e-06  bit-exact vs float64 0/576  ok
  wk       n= 192  max abs err 5.46e-06  max rel err 9.77e-06  bit-exact vs float64 0/192  ok
  w1       n=1536  max abs err 1.02e-06  max rel err 1.79e-05  bit-exact vs float64 0/1536  ok

same fp32 matmul (wq), different accumulation: bits identical to BASIC
  sequential fp32, round mul then add    576/576  max diff 0.00e+00
  sequential fp32, fused multiply-add    243/576  max diff 9.54e-07
  torch fp32 matmul (BLAS order)          61/576  max diff 1.91e-06
  float64 reference rounded to fp32       26/576  max diff 3.81e-06

FLOPs per token: matmul ~2.69e+08, rmsnorm ~1.05e+05 -> matmul share 99.9608

True

## Step 4: one transformer layer

Layer 0 run over a short token sequence, token *i* at position *i*. The reference for
the layer output is a forward hook on `model.model.layers[0]` at the last position; the
references for q and k after RoPE are an independent numpy implementation of the
interleaved rotation.

With a single token, attention is trivial (one score, softmax gives 1, the output is
just v), so the sequence is what actually exercises the attention loop, the GQA head
mapping, and RoPE making scores depend on the distance between positions.

In [6]:
compare_step4(model, tok, tok.encode("The cat sat on the", add_special_tokens=False))


compare BASIC layer 0 vs HF forward hook over 5 tokens ['The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe'], output at the last position
  criterion: |basic - ref| <= 0.0001 + 0.0001 * |ref|
  q_rope   n= 576  max abs err 4.04e-06  max rel err 9.03e-06  bit-exact vs float64 0/576  ok
  k_rope   n= 192  max abs err 4.52e-06  max rel err 6.17e-06  bit-exact vs float64 0/192  ok
  layer0   n= 576  max abs err 5.72e-06  max rel err 1.13e-04  bit-exact vs float64 23/576  ok
  residual stream at the last position: |x_in| rms 0.1159 -> |x_out| rms 1.2929
PASS


True

## Step 5: full forward and greedy decoding

All 30 layers, final norm, classifier. First the logits at the last prompt position are
compared with the real model, then greedy decoding is compared token for token against
`model.generate(do_sample=False)`.

`margin` is the gap between the best and second-best logit at each step: the numerical
headroom the greedy choice had. The BASIC engine re-forwards the whole sequence for every
new token (no KV cache yet), so watch the seconds per step grow linearly.

In [7]:
compare_step5(model, tok, tok.encode("The cat sat on the", add_special_tokens=False), 8)


full forward over ['The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe']: logits at the last position
  criterion: |basic - ref| <= 0.001 + 0.001 * |ref|
  logit 0: basic 7.290724  oracle 7.290715  diff 9.3e-06  ok
  logit 1: basic -2.707315  oracle -2.707330  diff 1.5e-05  ok
  logit 2: basic -2.664011  oracle -2.664025  diff 1.4e-05  ok
  logit 3: basic -4.295832  oracle -4.295849  diff 1.7e-05  ok
  logit 4: basic -3.879560  oracle -3.879569  diff 9.1e-06  ok
  top 1: basic id 4463 18.069260  oracle id 4463 18.069252  diff 8.0e-06  'Ġbed'  ok
  top 2: basic id 5595 17.590830  oracle id 5595 17.590834  diff 3.7e-06  'Ġedge'  ok
  top 3: basic id 5700 17.282160  oracle id 5700 17.282162  diff 1.7e-06  'Ġwindow'  ok
  top 4: basic id 3252 17.260630  oracle id 3252 17.260614  diff 1.6e-05  'Ġtable'  ok
  top 5: basic id 3187 16.982280  oracle id 3187 16.982281  diff 7.3e-07  'Ġtree'  ok



greedy decode, 8 tokens
  step 0: basic   4463 'Ġbed'         oracle   4463  logit diff 8.0e-06  margin 0.478  4.8s  ok
  step 1: basic     28 ','            oracle     28  logit diff 1.3e-05  margin 0.362  5.8s  ok
  step 2: basic    284 'Ġand'         oracle    284  logit diff 2.1e-06  margin 0.870  6.8s  ok
  step 3: basic    260 'Ġthe'         oracle    260  logit diff 3.0e-06  margin 0.344  7.7s  ok
  step 4: basic   2644 'Ġcat'         oracle   2644  logit diff 3.9e-06  margin 0.189  8.6s  ok
  step 5: basic   2643 'Ġsat'         oracle   2643  logit diff 1.6e-05  margin 0.937  9.7s  ok
  step 6: basic    335 'Ġon'          oracle    335  logit diff 1.4e-06  margin 2.667  10.6s  ok
  step 7: basic    260 'Ġthe'         oracle    260  logit diff 1.2e-05  margin 3.695  11.6s  ok
  smallest margin 0.189
  text: 'The cat sat on the' -> ' bed, and the cat sat on the'
  BASIC: 68 forwards in 65.7s, 0.12 tok/s
PASS


True